In [0]:
# Instalación de librería wquantiles para cálculo de medianas ponderadas
pip install wquantiles

In [0]:
# Reinicio del entorno Python para cargar las nuevas librerías instaladas
%restart_python
dbutils.library.restartPython()

In [0]:
# Importación de librerías para análisis estadístico y manipulación de datos
import pandas as pd
import numpy as np
import wquantiles
from scipy.stats import trim_mean

In [0]:
# Definición de rutas a los archivos de datos del libro
AIRLINE_STATS_CSV = '/Volumes/workspace/default/book/airline_stats.csv'
KC_TAX_CSV = '/Volumes/workspace/default/book/kc_tax.csv.gz'
LC_LOANS_CSV = '/Volumes/workspace/default/book/lc_loans.csv'
AIRPORT_DELAYS_CSV = '/Volumes/workspace/default/book/dfw_airline.csv'
SP500_DATA_CSV = '/Volumes/workspace/default/book/sp500_data.csv'
SP500_SECTORS_CSV = '/Volumes/workspace/default/book/sp500_sectors.csv'
STATE_CSV = '/Volumes/workspace/default/book/state.csv'

In [0]:
# Carga de datos de población y homicidios por estado
state = pd.read_csv(STATE_CSV)
print(state.head(8))

In [0]:
# Traducción de nombres de columnas al español
state.rename(columns={'State': 'estado','Population': 'poblacion', 'Murder.Rate': 'homicidios', 'Abbreviation': 'abreviacion'}, inplace=True)
print(state.head(8))

#### Medidas de Tendencia Central

`Media`
 - La media aritmética de un conjunto de valores se define como:
  $$ \bar{x} =  \frac{\sum_{i=1}^{n} x_i}{n} $$

`Mediana`
 - La mediana de un conjunto ordenado de n valores es:
  $$
  \tilde{x} = 
  \begin{cases}
  x_{\left(\frac{n+1}{2}\right)}, & \text{si } n \text{ es impar} \\
  \frac{x_{(\frac{n}{2})} + x_{(\frac{n}{2}+1)}}{2}, & \text{si } n \text{ es par}
  \end{cases}
  $$

`Media truncada`
 - La media truncada al eliminar k valores más pequeños y k valores más grandes:
  $$
  \bar{x}_{\text{trunc}} = \frac{\sum_{i=k+1}^{n-k} x_{(i)}}{n-2k} 
  $$

`Media ponderada`
 - La media ponderada de $n$ valores $x_i$ con pesos $w_i$:
  $$
  \bar{x}_w = \frac{\sum_{i=1}^{n} w_i x_i}{\sum_{i=1}^{n} w_i}
  $$

`Mediana ponderada`
 - La mediana ponderada es el valor $x_m$ tal que la suma de los pesos de los valores menores o iguales a $x_m$ es al menos la mitad del total de los pesos.
  $$
  \sum_{x_i \leq x_m} w_i \geq \frac{1}{2} \sum_{i=1}^{n} w_i
  $$

In [0]:
# Cálculo de diferentes medidas de tendencia central para homicidios
print('Media:', state['homicidios'].mean())
print('Mediana:', state['homicidios'].median())
print('Media truncada', trim_mean(state['homicidios'], 0.1))
print('Media ponderada:', np.average(state['homicidios'], weights=state['poblacion']))
print('Mediana ponderada:',wquantiles.median(state['homicidios'], weights=state['poblacion']))

#### Métricas de Variabilidad

`Desviación media absoluta`
  - La desviación media absoluta respecto a la media:
  $$
  \text{DMA} = \frac{\sum_{i=1}^{n} |x_i - \bar{x}|}{n} 
  $$

`Varianza`
  - La varianza muestral:
  $$
  s^2 = \frac{\sum_{i=1}^{n} (x_i - \bar{x})^2}{n-1} 
  $$

`Desviación estándar`
  - La desviación estándar muestral:
  $$
  s = \sqrt{\frac{\sum_{i=1}^{n} (x_i - \bar{x})^2}{n-1} }
  $$

`Desviación absoluta mediana de la mediana`
 - La desviación absoluta respecto a la mediana:
  $$
  \text{DAM} = \text{mediana}(|x_i - \tilde{x}|)
  $$

In [0]:
# Cálculo de métricas de variabilidad para la población
import statsmodels as st
from statsmodels import robust
# Desviación estándar
print('Desviación estandar: ', state['poblacion'].std())

# Varianza
print('Varianza: ', state['poblacion'].var())

# Coeficiente de variación
print(
    'Coeficiente de variación: ',
    state['poblacion'].std() / state['poblacion'].mean()
)

# Desviación absoluta mediana de la mediana
median = state['poblacion'].median()
mad_mediana = np.median(np.abs(state['poblacion'] - median))
print('Desviación absoluta mediana: ', mad_mediana)
print('MAD:', robust.scale.mad(state['poblacion']))
print('IQR: ', state['poblacion'].quantile(0.75) - state['poblacion'].quantile(0.25))

# Coeficiente de simetría
print('Coeficiente de simetría: ', state['poblacion'].skew())

# Coeficiente de curtosis
print('Coeficiente de curtosis: ', state['poblacion'].kurt())

In [0]:
# Cálculo de cuantiles específicos de la distribución de homicidios
state['homicidios'].quantile([0.05,.25,.5,.75,.95])

In [0]:
# Visualización de la distribución de población mediante diagrama de caja (boxplot)
ax = (state['poblacion']/1_000_000).plot.box()
ax.set_ylabel('Población (millones)')

In [0]:
# División de la población en 10 intervalos (bins) de igual ancho
binnedPopulation = pd.cut(state['poblacion'], 10)
binnedPopulation.value_counts()

In [0]:
# Creación de tabla que agrupa estados por rangos de población (Table 1.5)
binnedPopulation.name = 'binnedPopulation'
df = pd.concat([state, binnedPopulation], axis=1)
df = df.sort_values(by='poblacion')

groups = []
for group, subset in df.groupby(by='binnedPopulation', observed=False):
    groups.append({
        'BinRange': group,
        'Count': len(subset),
        'States': ','.join(subset.abreviacion)
    })
print(pd.DataFrame(groups))

In [0]:
# Histograma de la distribución de población por estado
ax = (state['poblacion']/1_000_000).plot.hist(figsize =(4,4))
ax.set_xlabel('Población (millones)')

In [0]:
# Histograma con curva de densidad superpuesta para homicidios
ax = state['homicidios'].plot.hist(
    density=True,
    xlim=[0, 12],
    bins=range(1, 12)
)
state['homicidios'].plot.density(ax=ax)
ax.set_xlabel('Homicidios (por cada 100,000 hab)')

#### Exploración de datos binarios y categóricos

`Moda:`
  - Categoría o valor que ocurre con más frecuencia en un conjunto de datos. 

`Valor esperado:`
  - Cuando las categorías se pueden asociar con un valor numérico, el valor esperado proporciona un valor promedio basado en la probabilidad de ocurrencia de una categoría. 

`Gráficos de barras:`
  - Frecuencia o proporción de cada categoría representada en barras. 
  
`Gráficos en forma de tarta:`
  - Frecuencia o proporción de cada categoría representada en forma de cuña de un pastel.

In [0]:
# Carga de datos de aerolíneas y visualización de retrasos por compañía
import matplotlib.pyplot as plt
AIRLINE_STATS_CSV = '/Volumes/workspace/default/book/airline_stats.csv'
airline_stats = pd.read_csv(AIRLINE_STATS_CSV)
airline_stats.head()
ax = airline_stats.boxplot(by='airline', column='pct_carrier_delay',
                           figsize=(5, 5))
ax.set_xlabel('')
ax.set_ylabel('Daily % of Delayed Flights')
plt.suptitle('')

plt.tight_layout()
plt.show()

In [0]:
# Gráfico de barras para comparar diferentes causas de retraso
ax = df.transpose().plot.bar(figsize=(4,4), legend=False)
ax.set_xlabel('Causas de retraso')
ax.set_ylabel('Recuento')

#### Probabilidad
la probabilidad de que suceda un evento es la proporción de veces que ocurriría si la situación pudiera repetirse una y otra vez, innumerables veces. La mayoría de las ocasiones se trata de una construcción imaginaria, pero es una comprensión funcional adecuada de la probabilidad.
displayHTML("<p style='color:red;'>Este texto es rojo</p>")
----------------------
#### Correlación

El análisis exploratorio de datos en muchos proyectos de modelado (ya sea en ciencia de datos o en investigación) implica examinar la correlación entre predictoras y entre predictoras y una variable objetivo. Se dice que las variables X e Y (cada una con datos registrados) están correlacionadas positivamente si los valores altos de X acompañan a los valores altos de Y, y los valores bajos de X acompañan a los valores bajos de Y. Si los valores altos de X acompañan a los valores bajos de Y, y viceversa, las variables están correlacionadas negativamente.

`Coeficiente de correlación`   
  - Métrica que mide el grado en que las variables numéricas están asociadas entre sí (varía de –1 a +1).

`Matriz de correlación`
 - Tabla en la que las variables se muestran tanto en filas como en columnas, y los valores de las celdas son las correlaciones entre las variables.

`Diagrama de dispersión`
 - Diagrama en el que el eje x es el valor de una variable y el eje y es el valor de otra.

Cálculo de `coeficiente de correlación de Pearson` (Pearson’s correlation coefficient) multiplicamos las desviaciones de la media de la variable 1 por las de la variable 2 y las dividimos por el producto de las desviaciones estándar:

$$
r = \frac{\sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})}{(n-1)s_xs_y} 
$$

El coeficiente de correlación siempre se encuentra entre +1 (correlación positiva perfecta) y –1 (correlación negativa perfecta). El 0 indica que no hay correlación.









In [0]:
# Carga de datos del S&P 500: símbolos con sectores y precios históricos
sp500_sym = pd.read_csv(
    "/Volumes/workspace/default/book/sp500_sectors.csv"
)
sp500_px = pd.read_csv(
    "/Volumes/workspace/default/book/sp500_data.csv",
    index_col=0
)
display(sp500_sym)
display(sp500_px)

In [0]:
# Filtración de datos de telecomunicaciones y cálculo de matriz de correlación (Table 1-7)
# Determine telecommunications symbols
telecomSymbols = sp500_sym[sp500_sym['sector'] == 'telecommunications_services']['symbol']

# Filter data for dates July 2012 through June 2015
telecom = sp500_px.loc[sp500_px.index >= '2012-07-01', telecomSymbols]
telecom.corr()
print(telecom)

In [0]:
# Filtración de datos de ETFs (Exchange Traded Funds) del S&P 500
etfs = sp500_px.loc[sp500_px.index > '2012-07-01', 
                    sp500_sym[sp500_sym['sector'] == 'etf']['symbol']]
print(etfs.head())

In [0]:
# Visualización de matriz de correlación mediante mapa de calor (heatmap)
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(
    figsize=[5, 4]
)
ax = sns.heatmap(
    etfs.corr(),
    vmin=-1,
    vmax=1,
    cmap=sns.diverging_palette(20, 220, as_cmap=True),
    ax=ax
)

plt.tight_layout()
plt.show()

In [0]:
# Función para visualizar correlaciones usando elipses y su aplicación a ETFs
from matplotlib.collections import EllipseCollection
from matplotlib.colors import Normalize

def plot_corr_ellipses(data, figsize=None, **kwargs):
    ''' https://stackoverflow.com/a/34558488 '''
    M = np.array(data)
    if not M.ndim == 2:
        raise ValueError('data must be a 2D array')
    fig, ax = plt.subplots(1, 1, figsize=figsize, subplot_kw={'aspect':'equal'})
    ax.set_xlim(-0.5, M.shape[1] - 0.5)
    ax.set_ylim(-0.5, M.shape[0] - 0.5)
    ax.invert_yaxis()

    # xy locations of each ellipse center
    xy = np.indices(M.shape)[::-1].reshape(2, -1).T

    # set the relative sizes of the major/minor axes according to the strength of
    # the positive/negative correlation
    w = np.ones_like(M).ravel() + 0.01
    h = 1 - np.abs(M).ravel() - 0.01
    a = 45 * np.sign(M).ravel()

    ec = EllipseCollection(widths=w, heights=h, angles=a, units='x', offsets=xy,
                           norm=Normalize(vmin=-1, vmax=1),
                           transOffset=ax.transData, array=M.ravel(), **kwargs)
    ax.add_collection(ec)

    # if data is a DataFrame, use the row/column names as tick labels
    if isinstance(data, pd.DataFrame):
        ax.set_xticks(np.arange(M.shape[1]))
        ax.set_xticklabels(data.columns, rotation=90)
        ax.set_yticks(np.arange(M.shape[0]))
        ax.set_yticklabels(data.index)

    return ec, ax

m, ax = plot_corr_ellipses(etfs.corr(), figsize=(5, 4), cmap='bwr_r')
cb = plt.colorbar(m, ax=ax)
cb.set_label('Correlation coefficient')

plt.tight_layout()
plt.show()

### Diagramas de dispersión

La forma tradicional de visualizar la relación entre dos variables de datos que hemos registrado es con un diagrama de dispersión.

In [0]:
# Diagrama de dispersión para comparar rendimientos de AT&T vs Verizon
ax = telecom.plot.scatter(x='T', y='VZ', figsize=(4, 4), marker='$◯$')
# ax = telecom.plot.scatter(x='T', y='VZ', figsize=(4, 4), marker='o')
ax.set_xlabel('ATT (T)')
ax.set_ylabel('Verizon (VZ)')
ax.axhline(0, color='grey', lw=1)
ax.axvline(0, color='grey', lw=1)

plt.tight_layout()
plt.show()

In [0]:
# Diagrama de dispersión con transparencia para visualizar concentración de datos
ax = telecom.plot.scatter(x='T', y='VZ', figsize=(4, 4), marker='$◯$', alpha=0.5)
ax.set_xlabel('ATT (T)')
ax.set_ylabel('Verizon (VZ)')
ax.axhline(0, color='grey', lw=1)
print(ax.axvline(0, color='grey', lw=1))

### Ideas Clave
- El coeficiente de correlación mide el grado en que dos variables emparejadas (por ejemplo, la altura y el peso de los individuos) están asociadas entre sí. 
- Cuando los valores altos de v1 acompañan a los valores altos de v2, v1 y v2 se asocian positivamente. 
- Cuando los valores altos de v1 acompañan a los valores bajos de v2, v1 y v2 se asocian negativamente. 
- El coeficiente de correlación es una métrica estandarizada, por lo que siempre varía de –1 (correlación negativa perfecta) a +1 (correlación positiva perfecta). 
- Un coeficiente de correlación de cero indica que no hay correlación, pero hay que tener en cuenta que las disposiciones aleatorias de datos producirán valores tanto positivos como negativos para el coeficiente de correlación simplemente por casualidad.

### Exploración de dos o más variables

Los estimadores habituales como la media y la varianza analizan las variables una por una (análisis univariable [univariate analysis]). 

El análisis de correlación es un método importante que compara dos variables (análisis bivariable [bivariate analysis]). 

En esta sección, analizamos estimaciones y diagramas complementarios aplicados a más de dos variables (análisis multivariante [multivariate analysis]).

----------------------

#### Términos clave de la exploración de dos o más variables

`Tabla de contingencia`
  - Registro del recuento entre dos o más variables categóricas. 

`Agrupación hexagonal`
  - Diagrama de dos variables numéricas con los registros agrupados en hexágonos. 

`Diagrama de contorno`
  - Diagrama que muestra la densidad de dos variables numéricas en forma de mapa topográfico. 

`Diagrama de violín`
  - Similar a un diagrama de caja pero en el que se muestra la densidad estimada.


In [0]:
KC_TAX_CSV = '/Volumes/workspace/default/book/kc_tax.csv.gz'

##### Agrupación hexagonal y contornos (representación numérica frente a datos numéricos)

Para conjuntos de datos con cientos de miles o millones de registros, un diagrama de dispersión será demasiado denso, por lo que necesitamos visualizar la relación de un modo diferente.

In [0]:
# Carga y filtrado de datos de impuestos del condado de King
kc_tax = pd.read_csv(KC_TAX_CSV)
#Filtra las propiedades que cumplen tres condiciones simultáneas:
#Valor de tasación (TaxAssessedValue) menor a $750,000
#Área habitable (SqFtTotLiving) mayor a 100 pies cuadrados
#Área habitable menor a 3,500 pies cuadrados
kc_tax0 = kc_tax.loc[(kc_tax.TaxAssessedValue < 750000) & 
                     (kc_tax.SqFtTotLiving > 100) &
                     (kc_tax.SqFtTotLiving < 3500), :]
print(kc_tax0.shape)

In [0]:
# Gráfico de agrupación hexagonal (hexbin) para visualizar relación área-valor
# el color indica la densidad de puntos (cantidad)
# el tamaño de los hexágonos indica el número de puntos que hay dentro
ax = kc_tax0.plot.hexbin(x='SqFtTotLiving', y='TaxAssessedValue',
                         gridsize=30, sharex=False, figsize=(5, 4))
ax.set_xlabel('Superficie terminada en pies cuadrados')
ax.set_ylabel('Valor de impuestos')

plt.tight_layout()
plt.show()

El _kdeplot_ de seaborn es una extensión bidimensional del gráfico de densidad. El cálculo de la densidad 2D para todo el conjunto de datos puede tomar varios minutos. Es suficiente crear la visualización con una muestra más pequeña del conjunto de datos. Con 10,000 puntos de datos, crear el gráfico toma solo unos segundos. Aunque se pueden perder algunos detalles, la forma general se conserva.

In [0]:
fig, ax = plt.subplots(figsize=(4, 4))
sns.kdeplot(data=kc_tax0.sample(20000), x='SqFtTotLiving', y='TaxAssessedValue', ax=ax)
ax.set_xlabel('Superficie terminada en pies cuadrados')
ax.set_ylabel('Valor de impuestos')

plt.tight_layout()
plt.show()

### Dos variables categóricas

Una forma conveniente de resumir dos variables categóricas es mediante una `tabla de contingencia, una tabla de recuentos por categorías`.

Esta información se ha extraído de los datos proporcionados por Lending Club, líder en el negocio de préstamos entre particulares. 

La calificación va desde A (la más alta) a G (la más baja). El resultado es: totalmente pagado, al corriente de pago, atrasado o cancelado (no se espera que se cobre el importe del préstamo). 

La tabla muestra el recuento y las filas de porcentajes. Los préstamos de alta calificación tienen un porcentaje muy bajo de retrasos en el pago/cancelaciones en comparación con los préstamos de baja calificación.

In [0]:
# Carga de la tabla lc_loans
lc_loans = pd.read_csv(LC_LOANS_CSV)

In [0]:
# Table 1-8(1)
# Tabla de contingencia: recuento de préstamos por calificación ('grade') y estado ('status')
crosstab = lc_loans.pivot_table(index='grade', columns='status', 
                                aggfunc=lambda x: len(x), margins=True)
print(crosstab)

In [0]:
# Table 1-8(2)
# Calcula porcentajes de préstamos por calificación y estado, normalizando por el total de cada calificación

# Copia la tabla de contingencia y selecciona solo las filas de calificación 'A' a 'G', convirtiendo los valores a float
df = crosstab.copy().loc['A':'G', :].astype(float)

# Normaliza las columnas 'Charged Off' a 'Late' dividiendo cada valor por el total de préstamos ('All') de cada calificación
df.loc[:, 'Charged Off':'Late'] = df.loc[:, 'Charged Off':'Late'].div(df['All'], axis=0)

# Calcula el porcentaje total de préstamos por calificación dividiendo cada valor de 'All' por la suma total de préstamos
df['All'] = df['All'] / sum(df['All'])
perc_crosstab = df
print(perc_crosstab)

En las `tablas de contingencia` solo se suelen ver recuentos, aunque también pueden incluir porcentajes totales y de cada columna. Las tablas dinámicas en Excel son quizá la herramienta más utilizada para crear tablas de contingencia. 